# ECAPA-TDNN fine-tuning on Google Colab
Run the sanity subset first. The full-training cell is intentionally commented out.

In [ ]:
# Cell 1 — Check GPU
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2 — Clone repo
!git clone -b feature/finetune-colab https://github.com/hieuvous/Secure-Virtual-Assistant-with-Speaker-Recognition.git
%cd /content/Secure-Virtual-Assistant-with-Speaker-Recognition

In [ ]:
# Cell 3 — Install only fine-tuning/evaluation requirements. Colab already provides PyTorch.
!pip install -q speechbrain==1.1.0 torchaudio pandas scikit-learn soundfile

In [ ]:
# Cell 4 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 5 — Dataset, metadata and cache are temporary in /content. Drive stores only models/checkpoints.
DATASET_ROOT = '/content/Vietnam-Celeb/data'
OFFICIAL_TRAIN_LIST = '/content/Vietnam-Celeb/vietnam-celeb-t.txt'
METADATA_DIR = '/content/SpeakerRecognition/metadata_small'
CHECKPOINT_DIR = '/content/drive/MyDrive/SpeakerRecognition/checkpoints_small'
OUTPUT_MODEL = f'{CHECKPOINT_DIR}/best_model.pt'  # Same path: no duplicate best-model file.
CACHE_DIR = '/content/pretrained_ecapa_cache'

In [ ]:
# Cell 6 — Prepare small subset: 20 train speakers, 5 dev speakers, max 10 utterances.
!python training/prepare_vietnam_celeb_subset.py --data-root "$DATASET_ROOT" --official-train-list "$OFFICIAL_TRAIN_LIST" --output-dir "$METADATA_DIR" --train-speakers 20 --dev-speakers 5 --max-utts 10 --seed 42

In [ ]:
# Cell 7 — Inspect metadata
import pandas as pd
for name in ('train', 'val', 'dev'):
    df = pd.read_csv(f'{METADATA_DIR}/{name}.csv')
    print(f'\n{name}.csv: speakers={df.speaker_id.nunique()}, audio={len(df)}')
    display(df.head())

In [ ]:
# Cell 8 — Sanity fine-tune: 1 epoch, batch size 8.
!python training/finetune_ecapa_colab.py \
  --train-csv "$METADATA_DIR/train.csv" \
  --val-csv "$METADATA_DIR/val.csv" \
  --output "$OUTPUT_MODEL" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --cache-dir "$CACHE_DIR" \
  --epochs 1 \
  --batch-size 8 \
  --lr 1e-4

In [ ]:
# Cell 9 — Full fine-tune (run only after sanity passes).
# Change these paths to a separate full-experiment metadata/checkpoint directory first.
# !python training/prepare_vietnam_celeb_subset.py --data-root "$DATASET_ROOT" --official-train-list "$OFFICIAL_TRAIN_LIST" --output-dir /content/SpeakerRecognition/metadata_full --train-speakers 300 --dev-speakers 50 --max-utts 20 --seed 42
# !python training/finetune_ecapa_colab.py --train-csv /content/SpeakerRecognition/metadata_full/train.csv --val-csv /content/SpeakerRecognition/metadata_full/val.csv --dev-csv /content/SpeakerRecognition/metadata_full/dev.csv --output /content/drive/MyDrive/SpeakerRecognition/checkpoints_full/best_model.pt --checkpoint-dir /content/drive/MyDrive/SpeakerRecognition/checkpoints_full --cache-dir "$CACHE_DIR" --epochs 25 --batch-size 16 --lr 1e-4 --patience 5 --eval-every 5 --eval-max-utts-per-speaker 8

In [ ]:
# Cell 10 — Resume example: latest_checkpoint.pt continues at its next epoch.
# !python training/finetune_ecapa_colab.py --train-csv "$METADATA_DIR/train.csv" --val-csv "$METADATA_DIR/val.csv" --output "$OUTPUT_MODEL" --checkpoint-dir "$CHECKPOINT_DIR" --cache-dir "$CACHE_DIR" --epochs 5 --batch-size 8 --lr 1e-4 --resume "$CHECKPOINT_DIR/latest_checkpoint.pt"

In [ ]:
# Cell 11 — Evaluation: pretrained and fine-tuned on the identical dev split.
PRETRAINED_METRICS = '/content/SpeakerRecognition/pretrained_metrics.json'
FINETUNED_METRICS = '/content/SpeakerRecognition/finetuned_metrics.json'
!python training/evaluate_verification.py --dev-csv "$METADATA_DIR/dev.csv" --output-json "$PRETRAINED_METRICS" --cache-dir "$CACHE_DIR" --max-utts-per-speaker 8 --seed 42
!python training/evaluate_verification.py --dev-csv "$METADATA_DIR/dev.csv" --checkpoint "$OUTPUT_MODEL" --output-json "$FINETUNED_METRICS" --cache-dir "$CACHE_DIR" --max-utts-per-speaker 8 --seed 42

In [ ]:
# Cell 12 — Show comparison
import json
rows = []
for label, path in [('Pretrained', PRETRAINED_METRICS), ('Fine-tuned', FINETUNED_METRICS)]:
    with open(path) as f: m = json.load(f)
    rows.append({'Model': label, 'EER': m['eer'], 'FAR': m['far'], 'FRR': m['frr'], 'Threshold': m['threshold']})
display(pd.DataFrame(rows))